# Trabalho de Séries Temporais — Pipeline final (Grupo 4 · MLP Regressor / PyTorch)

**Base:** PPC (Pilgrim's Pride) com variáveis externas de mercado · **Alvo:** `Close`

Este notebook cobre todas as etapas obrigatórias do enunciado para **uma base**, executando os
**4 modelos**: SARIMAX, Holt-Winters, Random Forest e **MLP Regressor (PyTorch)**, com o mesmo protocolo
(mesmas origens, mesmo horizonte, mesmo conjunto de teste).

| Seção | Item do enunciado |
|---|---|
| 1–3 | 5.1 Documentação e exploração da base · 4.1 Disponibilidade das variáveis |
| 4 | 5.2 STL, tendência e força da sazonalidade |
| 5 | 5.3 Feature Engineering sem vazamento |
| 6 | 5.4 Protocolo walk-forward |
| 7 | 5.5 Otimização de hiperparâmetros (sem usar o teste) |
| 8 | 5.6 Treinamento e previsão (tempo, parâmetros, previsões, resíduos) |
| 9 | 6 Comparação por MAE · 9.1 significância estatística · 9.2 ablação das exógenas |
| 10 | 7 Análise dos resíduos (ACF + Ljung-Box) · 10.1 análise por regime |
| 11 | 8 Importância das features |
| 12 | 9 Estudo do modelo de especialização (MLP) |
| 13 | Consolidação entre bases (vitórias e posição média) |

> **Para rodar em outra base:** altere apenas a célula de configuração (seção 0).

## 0. Imports e configuração

In [ ]:
import warnings, time, json, os, random, itertools, re
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import torch
import torch.nn as nn

import pmdarima as pm          # pip install pmdarima  (auto_arima, ndiffs, nsdiffs)
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams["figure.dpi"] = 100
print("Device:", device)

In [ ]:
# ====================== CONFIGURAÇÃO DA BASE (mude só este bloco para outras bases) ======================
BASE_NAME = "PPC"
DATA_PATH = "../datasets/PPC_exogenous_2013-07-25_2026-09-16.csv"
DATE_COL  = "Date"
TARGET    = "Close"
EXOG_COLS = ["TSN_Close", "XLP_Close", "CALM_Close", "USD_MXN"]   # variáveis externas
USE_LOG   = True      # True: séries de preço (usa log-retorno). False: séries que podem ser <= 0 (usa diferença)

# ---- protocolo (IDÊNTICO para os 4 modelos) ----
H          = 5        # horizonte: prever os próximos 5 pregões (1 semana) a partir de cada origem
STEP       = 5        # avanço entre origens (= H  ->  cada dia do teste é previsto exatamente uma vez)
N_TEST     = 500      # últimos N pregões = conjunto de TESTE (nunca usado para decidir nada)
SEASONAL_M = 5        # período sazonal candidato (semana de pregão)

# ---- tuning menos ruidoso: validação maior e mais densa + triagem em 2 etapas ----
N_VAL        = 500    # N pregões imediatamente antes do teste = janela de VALIDAÇÃO
STEP_VAL     = 10     # espaçamento das origens de validação (~50 origens)
SCREEN_EVERY = 3      # etapa 1 (triagem): usa 1 a cada 3 origens de validação
N_KEEP       = 3      # finalistas que passam à etapa 2 (validação completa, c/ erro-padrão)

# ---- escala e exógenas dos modelos estatísticos ----
LOG_SX_HW    = True          # SARIMAX e Holt-Winters modelam log(Close) (como RF/MLP, que preveem log-retorno)
SX_EXOG_MODE = "equalizado"  # "equalizado": mesmas famílias de features de RF/MLP (defasadas em H) | "simples": só retorno 1d

# ---- otimização / treino ----
N_ITER_RF, N_ITER_MLP, N_TOP_SARIMAX = 16, 12, 3
MLP_EPOCHS_TUNE, MLP_EPOCHS_FINAL, MLP_PATIENCE = 150, 300, 20
N_SEEDS_TUNE    = 2   # redes por configuração na etapa 2 do tuning (média de sementes)
N_SEEDS_FINAL   = 5   # redes no MLP final: a previsão é a MÉDIA de N redes com sementes diferentes
REFIT_EVERY_TAB = 1   # re-treina RF/MLP a cada k origens (1 = todas, como no enunciado; use 4 para ~4x mais rápido)
RODAR_ABLACAO   = True  # roda também RF, MLP e SARIMAX SEM as exógenas (dobra o custo do teste)
BOOT_B          = 5000  # repetições do bootstrap por blocos (testes de significância)
USE_CACHE       = False
RESULTS_DIR     = "resultados"

QUICK_MODE = False    # True = teste de fumaça rápido (poucos dados/iterações). NÃO use para o relatório.
if QUICK_MODE:
    N_TEST, N_VAL, STEP_VAL, SCREEN_EVERY, N_KEEP = 100, 126, 21, 2, 2
    N_ITER_RF, N_ITER_MLP, N_TOP_SARIMAX = 3, 3, 2
    MLP_EPOCHS_TUNE, MLP_EPOCHS_FINAL, N_SEEDS_TUNE, N_SEEDS_FINAL, BOOT_B = 30, 40, 1, 2, 500
    RESULTS_DIR = "resultados_quick"

MODEL_LOG = bool(LOG_SX_HW and USE_LOG)   # escala em que SARIMAX/Holt-Winters são ajustados

os.makedirs(RESULTS_DIR, exist_ok=True)
if STEP != H:
    print("ATENÇÃO: STEP != H -> os resíduos concatenados não formam uma série contígua (afeta ACF/Ljung-Box).")

## 1. Documentação e exploração da base (item 5.1)

- **Fonte (a confirmar pelo grupo):** dados diários de mercado no formato do Yahoo Finance (`yfinance`) — o grupo deve
  citar a fonte exata usada na base congelada.
- **Alvo:** `Close` — preço de fechamento da PPC, em **USD por ação**, frequência de **pregão** (dias úteis com bolsa aberta).
- **Cobertura:** 25/07/2013 a 16/09/2026.

In [ ]:
raw = pd.read_csv(DATA_PATH)
raw[DATE_COL] = pd.to_datetime(raw[DATE_COL], utc=True).dt.tz_localize(None).dt.normalize()
raw = raw.sort_values(DATE_COL).set_index(DATE_COL)
print("Formato bruto:", raw.shape, "|", raw.index.min().date(), "->", raw.index.max().date())

# ---- 1) duplicidades e ausentes ----
n_dup = raw.index.duplicated().sum()
print("Datas duplicadas:", n_dup)
raw = raw[~raw.index.duplicated(keep="last")]
print("Valores ausentes por coluna:\n", raw.isna().sum().to_string())

# ---- 2) irregularidades temporais ----
gaps = raw.index.to_series().diff().dt.days
print("\nIntervalo entre linhas (dias corridos):", gaps.value_counts().sort_index().to_dict())
print("Linhas em fim de semana:", int((raw.index.dayofweek >= 5).sum()))

# ---- 3) dias sem pregão preenchidos artificialmente ----
# Feriados da bolsa aparecem com Volume = 0 e preços copiados do dia anterior (forward-fill).
if "Volume" in raw.columns:
    stale = (raw["Volume"] <= 0) & (raw[TARGET] == raw[TARGET].shift(1))
else:
    stale = pd.Series(False, index=raw.index)
print(f"\nLinhas de dias SEM pregão (Volume=0 e preço repetido): {int(stale.sum())} de {len(raw)}")
print("Exemplos:", [d.strftime("%Y-%m-%d") for d in raw.index[stale][:6]])

**Decisão de limpeza #1 — remover os dias sem pregão.** Essas linhas não são observações reais: repetem o preço do dia
anterior e produzem retornos exatamente zero, o que subestima a volatilidade e cria um padrão falso (a rede/árvore
"aprenderia" que às vezes o preço não muda). Removemos e passamos a trabalhar com o **calendário de pregões**
(a frequência regular passa a ser "dia de pregão", período semanal ≈ 5).

In [ ]:
df = raw.loc[~stale].copy()
df = df.ffill()                       # (não há ausentes nesta base; mantido por segurança)
print("Base após limpeza:", df.shape, "|", df.index.min().date(), "->", df.index.max().date())
print("Intervalos (dias corridos):", df.index.to_series().diff().dt.days.value_counts().sort_index().to_dict())
assert (df[[TARGET] + EXOG_COLS] > 0).all().all() or not USE_LOG, "USE_LOG=True exige valores positivos"

# ---- 4) valores atípicos (z robusto sobre retornos de 1 dia) ----
def z_robusto(s):
    med = s.median(); mad = (s - med).abs().median()
    return (s - med) / (1.4826 * mad)

chg1 = (lambda s: np.log(s).diff()) if USE_LOG else (lambda s: s.diff())
cols_check = [TARGET] + EXOG_COLS
outl = {c: int((z_robusto(chg1(df[c]).dropna()).abs() > 6).sum()) for c in cols_check}
print("\nRetornos com |z robusto| > 6 por variável:", outl)
top = z_robusto(chg1(df[TARGET]).dropna()).abs().sort_values(ascending=False).head(6)
print("\nMaiores retornos atípicos do alvo:")
print(pd.DataFrame({"data": top.index.date, "|z|": top.values.round(1),
                    "retorno": chg1(df[TARGET]).loc[top.index].values.round(3)}).to_string(index=False))
df.describe().T

**Decisão de limpeza #2 — manter os outliers.** Os retornos extremos correspondem a eventos reais de mercado (resultados
trimestrais, choques setoriais, 2020). Removê-los faria o modelo subestimar o risco real; por isso são **mantidos**.
A robustez é tratada pela escolha da métrica (MAE, menos sensível a extremos que o MSE) e pelo `L1Loss` na rede.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
ax[0].plot(df.index, df[TARGET]); ax[0].set_title(f"{TARGET} (USD/ação) — série-alvo")
base100 = df[EXOG_COLS] / df[EXOG_COLS].iloc[0] * 100
for c in EXOG_COLS: ax[1].plot(df.index, base100[c], label=c, lw=1)
ax[1].set_title("Variáveis externas (base 100 = início)"); ax[1].legend(ncol=4, fontsize=8)
if "Volume" in df.columns:
    ax[2].plot(df.index, df["Volume"], lw=0.6, color="darkorange"); ax[2].set_title("Volume")
plt.tight_layout(); plt.show()

corr = pd.concat([chg1(df[c]) for c in cols_check], axis=1, keys=cols_check).corr()
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(cols_check))); ax.set_xticklabels(cols_check, rotation=45, ha="right")
ax.set_yticks(range(len(cols_check))); ax.set_yticklabels(cols_check)
for i in range(len(cols_check)):
    for j in range(len(cols_check)): ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlação entre retornos diários"); plt.colorbar(im); plt.tight_layout(); plt.show()

## 2. Dicionário das variáveis e disponibilidade na data da previsão (item 4.1)

**Regra adotada:** a origem `t` é o **fechamento do pregão `t`**. Tudo o que foi observado até `t` está disponível; nada
de `t+1` em diante pode entrar. Como o modelo prevê **vários dias à frente** (`t+1 … t+H`), uma exógena só pode ser usada
com **defasagem** em relação ao dia previsto.

In [ ]:
dicionario = pd.DataFrame([
    ["Close",      "Preço de fechamento da PPC (alvo)",                       "USD/ação",   "Alvo",     "Histórico até t: sim. Futuro: NÃO",          "Só lags/retornos passados"],
    ["Open",       "Preço de abertura da PPC",                                "USD/ação",   "Própria",  "Até t: sim. Futuro: NÃO",                     "Gap de abertura em t (Open_t / Close_{t-1})"],
    ["High / Low", "Máxima / mínima do dia da PPC",                           "USD/ação",   "Própria",  "Só completas no fechamento de t",             "Amplitude intradia em t (log High/Low)"],
    ["Volume",     "Quantidade negociada da PPC",                             "ações/dia",  "Própria",  "Só completo no fechamento de t",              "Volume relativo à média de 21 dias, em t"],
    ["TSN_Close",  "Fechamento da Tyson Foods (concorrente direta, frango)",  "USD/ação",   "Externa",  "Até t: sim. Futuro: NÃO",                     "Retornos em t, t-1, 5d, 21d; SARIMAX: retorno defasado H"],
    ["XLP_Close",  "ETF de bens de consumo básico (mercado defensivo)",       "USD/cota",   "Externa",  "Até t: sim. Futuro: NÃO",                     "Idem"],
    ["CALM_Close", "Fechamento da Cal-Maine Foods (ovos/proteína)",           "USD/ação",   "Externa",  "Até t: sim. Futuro: NÃO",                     "Idem"],
    ["USD_MXN",    "Câmbio dólar/peso mexicano (PPC opera no México)",        "MXN por USD","Externa",  "Até t: sim. Futuro: NÃO (nem previsão na base)", "Idem"],
], columns=["variável", "descrição", "unidade", "papel", "disponibilidade na origem", "uso no modelo"])
pd.set_option("display.max_colwidth", 70); dicionario

> Nenhuma das exógenas é conhecida com antecedência (não há calendário/plano). Por isso **nenhum valor futuro observado**
> é usado: os modelos tabulares enxergam as exógenas **até a origem**, e o SARIMAX recebe as exógenas **defasadas em `H`**
> (para prever `t+k`, usa-se o valor de `t+k-H ≤ t`). Isso é o que evita o vazamento descrito na seção 4.1 do enunciado.

## 3. Partição temporal: treino · validação · teste

```
|----------------------- treino (expande) -----------------------|-- validação (N_VAL) --|---- teste (N_TEST) ----|
                                                                    ^ tuning aqui           ^ avaliação final (MAE)
```
Toda decisão de hiperparâmetros usa apenas dados **anteriores ao teste**.

In [ ]:
N = len(df)
TEST_START = N - N_TEST                 # primeiro dia previsto no teste
VAL_START  = TEST_START - N_VAL
y     = df[TARGET].values.astype(float)
dates = df.index

# origem o => dados até o (inclusive); previsões para o+1 .. o+H
ORIGINS_TEST = list(range(TEST_START - 1, N - H, STEP))
ORIGINS_VAL  = list(range(VAL_START - 1, TEST_START - H, STEP_VAL))   # alvos sempre < TEST_START

print(f"N={N} | treino+val até {dates[TEST_START-1].date()} | teste {dates[TEST_START].date()} -> {dates[-1].date()}")
print(f"Origens de validação: {len(ORIGINS_VAL)} ({dates[ORIGINS_VAL[0]].date()} … {dates[ORIGINS_VAL[-1]].date()})")
print(f"Origens de teste:     {len(ORIGINS_TEST)} ({dates[ORIGINS_TEST[0]].date()} … {dates[ORIGINS_TEST[-1]].date()})")
assert ORIGINS_VAL[-1] + H <= TEST_START - 1, "vazamento: validação enxerga o teste"

## 4. Decomposição STL, tendência e força da sazonalidade (item 5.2)

A STL é **descritiva** e escolhe o período `m`; por isso é calculada **apenas com dados anteriores ao teste**.
Força da sazonalidade (Hyndman): $F_S = \max\left(0,\ 1 - \frac{\mathrm{Var}(R)}{\mathrm{Var}(S+R)}\right)$, e analogamente
$F_T$ para a tendência. Valores próximos de 0 = sem sazonalidade; próximos de 1 = sazonalidade forte.

In [ ]:
y_pre = pd.Series(y[:TEST_START], index=dates[:TEST_START], name=TARGET)

def forca(res, comp):
    c = getattr(res, comp)
    return float(max(0.0, 1 - res.resid.var() / (c + res.resid).var()))

# varredura do período m (como no notebook base do SARIMAX)
periodos = list(range(2, 101)) + [126, 252]
linhas = []
for m in periodos:
    r = STL(y_pre, period=m, robust=True).fit()
    linhas.append((m, forca(r, "seasonal"), forca(r, "trend")))
df_forca = pd.DataFrame(linhas, columns=["m", "forca_sazonal", "forca_tendencia"]).sort_values("forca_sazonal", ascending=False)
print("Maiores forças sazonais:"); print(df_forca.head(8).round(4).to_string(index=False))
print("\nPeríodos de referência:"); print(df_forca[df_forca["m"].isin([5, 21, 63, 126, 252])].sort_values("m").round(4).to_string(index=False))

M_BEST = int(df_forca.iloc[0]["m"])
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.bar(df_forca.sort_values("m")["m"], df_forca.sort_values("m")["forca_sazonal"], width=0.8)
ax.axvline(SEASONAL_M, color="red", ls="--", label=f"m={SEASONAL_M} (semana de pregão)"); ax.legend()
ax.set_xlabel("período sazonal m"); ax.set_ylabel("força sazonal"); ax.set_title("Força da sazonalidade por período (STL)")
plt.tight_layout(); plt.show()

In [ ]:
M_STL = SEASONAL_M                     # período usado na decomposição exibida (semana de pregão)
stl = STL(y_pre, period=M_STL, robust=True).fit()
fig = stl.plot(); fig.set_size_inches(12, 8); plt.tight_layout(); plt.show()
print(f"m={M_STL}: força sazonal = {forca(stl, 'seasonal'):.4f} | força da tendência = {forca(stl, 'trend'):.4f}")
print(f"(m com maior força sazonal: {M_BEST} -> {df_forca.iloc[0]['forca_sazonal']:.4f})")

# ---- segmentação da tendência: crescimento / queda / estabilidade ----
def segmentar_tendencia(trend, janela=63, limiar=0.05, min_len=21):
    var = np.log(trend.clip(lower=1e-9)).diff(janela)
    est = pd.Series(np.where(var > limiar, "crescimento", np.where(var < -limiar, "queda", "estabilidade")), index=trend.index)
    est[var.isna()] = np.nan
    grp = (est != est.shift()).cumsum()
    segs = []
    for _, s in est.groupby(grp):
        if s.isna().all() or len(s) < min_len: continue
        ini, fim = s.index[0], s.index[-1]
        segs.append(dict(inicio=ini.date(), fim=fim.date(), regime=s.iloc[0], pregoes=len(s),
                         var_tendencia_pct=round((trend[fim] / trend[ini] - 1) * 100, 1)))
    return pd.DataFrame(segs), est

segs, estado = segmentar_tendencia(stl.trend)
cores = {"crescimento": "#c8e6c9", "queda": "#ffcdd2", "estabilidade": "#eeeeee"}
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_pre.index, y_pre, color="lightgray", lw=0.7, label="Close"); ax.plot(stl.trend.index, stl.trend, color="black", lw=1.6, label="tendência STL")
for _, r in segs.iterrows(): ax.axvspan(pd.Timestamp(r["inicio"]), pd.Timestamp(r["fim"]), color=cores[r["regime"]], alpha=0.6, lw=0)
ax.set_title("Tendência STL por regime (verde=crescimento, vermelho=queda, cinza=estabilidade)"); ax.legend(loc="upper left"); plt.tight_layout(); plt.show()
segs

**Como interpretar (preencher no relatório com o que aparecer nos gráficos/tabelas acima):**
- **Tendência:** liste os regimes da tabela (datas, variação %) e as mudanças de direção; relacione com eventos do setor.
- **Sazonal:** se a força sazonal for baixa (típico de preço de ação), justifique que o ganho de modelos sazonais é limitado
  e que o Holt-Winters/SARIMAX sazonais devem competir com versões sem sazonalidade (estão no espaço de busca).
- **Resíduo:** concentra quase toda a variância → série dominada por ruído/choques; o poder preditivo esperado é modesto.

### 4.1 Estacionariedade (define o `d` do SARIMAX) — ADF + KPSS em dados pré-teste

In [ ]:
def estacionaria(s, nome):
    s = pd.Series(s).dropna()
    adf_p  = adfuller(s, autolag="AIC")[1]
    kpss_p = kpss(s, regression="c", nlags="auto")[1]
    ok = (adf_p <= 0.05) and (kpss_p > 0.05)
    print(f"{nome:<22} ADF p={adf_p:.4f} | KPSS p={kpss_p:.4f} -> {'estacionária' if ok else 'NÃO estacionária'}")
    return ok

y_pre_m = np.log(y_pre) if MODEL_LOG else y_pre     # escala em que SARIMAX/Holt-Winters são ajustados
print('Escala usada pelo SARIMAX/Holt-Winters:', 'log(Close)' if MODEL_LOG else 'Close')
D_SEL, s_tmp = 0, y_pre_m.copy()
while not estacionaria(s_tmp, f"série (d={D_SEL})") and D_SEL < 2:
    s_tmp = s_tmp.diff().dropna(); D_SEL += 1
print("\nOrdem de diferenciação escolhida (d):", D_SEL)

# ---- verificação cruzada com pmdarima: ndiffs (d) e nsdiffs (D sazonal) ----
d_adf  = pm.arima.ndiffs(y_pre_m.values, test="adf")
d_kpss = pm.arima.ndiffs(y_pre_m.values, test="kpss")
D_sazonal = {}
for t_ in ("ocsb", "ch"):
    try:    D_sazonal[t_] = pm.arima.nsdiffs(y_pre_m.values, m=SEASONAL_M, test=t_)
    except Exception as e: D_sazonal[t_] = f"falhou ({str(e)[:40]})"
print(f"\npmdarima.ndiffs  -> ADF: d={d_adf} | KPSS: d={d_kpss}   (ADF+KPSS manual: d={D_SEL})")
print(f"pmdarima.nsdiffs (m={SEASONAL_M}) -> D por teste: {D_sazonal}")

## 5. Feature Engineering (item 5.3) — compartilhada por Random Forest e MLP

**Formulação (direct multi-horizon):** em cada origem `t`, as features usam **somente informação até `t`** e o alvo é o vetor
`[Close_{t+1}, …, Close_{t+H}]`, escrito como **log-retorno acumulado** `log(Close_{t+k}/Close_t)`. A previsão em nível
é `Close_t · exp(previsão)`. Isso (i) elimina a não-estacionariedade do nível, (ii) permite que árvores/redes
prevejam valores fora do intervalo visto no treino (o preço subiu de ~10 para ~50) e (iii) mantém o MAE em USD.

**Anti-vazamento no treino:** a linha de treino `i` só entra na origem `o` se `i + H ≤ o` (seu alvo já foi observado).

| Grupo | Features |
|---|---|
| Histórico do alvo | retornos de 1 dia (lags 0–4), retornos acumulados 5/10/21/63d, volatilidade 5/21/63d, distância à média móvel 5/21/63d |
| Própria (OHLCV) | gap de abertura, amplitude High/Low, volume relativo |
| **Externas** | retorno 1d (lag 0 e 1), 5d e 21d de cada exógena |
| Calendário | dia da semana e mês com encoding cíclico (seno/cosseno) |

In [ ]:
chg = (lambda s, k=1: np.log(s).diff(k)) if USE_LOG else (lambda s, k=1: s.diff(k))
make_target  = (lambda cur, fut: np.log(fut / cur)) if USE_LOG else (lambda cur, fut: fut - cur)
back_transf  = (lambda cur, pred: cur * np.exp(pred)) if USE_LOG else (lambda cur, pred: cur + pred)

def build_features(d):
    yy = d[TARGET]; r1 = chg(yy, 1); F = {}
    for L in range(5): F[f"ret1_lag{L}"] = r1.shift(L)
    for k in (5, 10, 21, 63): F[f"ret{k}d"] = chg(yy, k)
    for W in (5, 21, 63):
        F[f"vol{W}d"] = r1.rolling(W).std()
        F[f"dist_ma{W}"] = (np.log(yy / yy.rolling(W).mean()) if USE_LOG else yy - yy.rolling(W).mean())
    if USE_LOG:
        if {"High", "Low"} <= set(d.columns): F["range_hl"] = np.log(d["High"] / d["Low"])
        if "Open" in d.columns:               F["gap_open"] = np.log(d["Open"] / yy.shift(1))
        if "Volume" in d.columns:             F["vol_rel21"] = np.log((d["Volume"] + 1) / (d["Volume"].rolling(21).mean() + 1))
    for c in EXOG_COLS:
        rc = chg(d[c], 1)
        F[f"{c}_ret1_lag0"], F[f"{c}_ret1_lag1"] = rc, rc.shift(1)
        F[f"{c}_ret5d"], F[f"{c}_ret21d"] = chg(d[c], 5), chg(d[c], 21)
    dow, mes = d.index.dayofweek.values, d.index.month.values
    F["dow_sin"], F["dow_cos"] = np.sin(2*np.pi*dow/5),  np.cos(2*np.pi*dow/5)
    F["mes_sin"], F["mes_cos"] = np.sin(2*np.pi*mes/12), np.cos(2*np.pi*mes/12)
    return pd.DataFrame(F, index=d.index)

F = build_features(df)
FEATURES = list(F.columns)
F_arr = F.values
VALID = F.notna().all(axis=1).values
FIRST_VALID = int(np.argmax(VALID))
print(f"{len(FEATURES)} features | linhas descartadas pelo aquecimento de lags/janelas: {FIRST_VALID} (de {N})")

def grupo(nome):
    if any(nome.startswith(c) for c in EXOG_COLS): return "externa"
    if nome in ("range_hl", "gap_open", "vol_rel21"): return "própria (OHLCV)"
    if nome.startswith(("dow_", "mes_")): return "calendário"
    return "histórico do alvo"
GRUPO = {f: grupo(f) for f in FEATURES}
print(pd.Series(GRUPO).value_counts().to_dict())

# matriz de alvos multi-horizonte (em log-retorno acumulado)
Y = np.full((N, H), np.nan)
for k in range(1, H + 1):
    Y[:N - k, k - 1] = make_target(y[:N - k], y[k:])

ROW = np.arange(N)
def train_rows(o):
    """linhas de treino disponíveis na origem o: alvo totalmente observado (i+H <= o) e features válidas."""
    return np.where(VALID & (ROW <= o - H))[0]

assert train_rows(ORIGINS_TEST[0]).max() + H <= ORIGINS_TEST[0]
print("Linhas de treino na 1ª origem de teste:", len(train_rows(ORIGINS_TEST[0])))

## 6. Protocolo walk-forward (item 5.4)

Para cada origem `o` (fechamento do dia `o`): o modelo é treinado **só com dados ≤ `o`**, prevê `o+1 … o+H`; a origem avança
`STEP` pregões, os dados reais entram no histórico e o modelo é re-treinado. **Todos os modelos usam exatamente
`ORIGINS_TEST`, o mesmo `H` e o mesmo teste.** Hiperparâmetros são escolhidos antes (seção 7) e ficam **fixos** aqui.

Todos os modelos expõem a mesma interface: `forecaster(o, j) -> array de H previsões em USD`.

In [ ]:
def run_walk_forward(forecast_fn, origins, nome="", verbose=True):
    rows, t0 = [], time.time()
    for j, o in enumerate(origins):
        pred = np.asarray(forecast_fn(o, j), dtype=float).ravel()
        assert len(pred) == H and o + H < N
        for k in range(1, H + 1):
            rows.append((dates[o], dates[o + k], k, y[o], y[o + k], pred[k - 1]))
        if verbose and (j + 1) % 20 == 0:
            print(f"   {nome}: {j+1}/{len(origins)} origens ({time.time()-t0:.0f}s)")
    out = pd.DataFrame(rows, columns=["origem", "data", "passo", "close_origem", "real", "pred"])
    out["erro"] = out["real"] - out["pred"]
    return out, time.time() - t0

def mae_of(d): return float(d["erro"].abs().mean())
def mae_por_origem(d): return d.assign(ae=d["erro"].abs()).groupby("origem")["ae"].mean().sort_index()
def mae_se(d):
    m = mae_por_origem(d); return float(m.std(ddof=1) / np.sqrt(len(m)))      # erro-padrão do MAE entre origens

def amostrar(space, n, seed=SEED):
    """busca aleatória: n configurações distintas sorteadas do espaço."""
    rng, seen, out = random.Random(seed), set(), []
    for _ in range(n * 50):
        cfg = {k: rng.choice(v) for k, v in space.items()}
        key = json.dumps(cfg, sort_keys=True, default=str)
        if key not in seen: seen.add(key); out.append(cfg)
        if len(out) == n: break
    return out

# escala em que SARIMAX e Holt-Winters são ajustados (log(Close) -> volta a USD com exp)
Y_M = np.log(y) if MODEL_LOG else y
INV = np.exp if MODEL_LOG else (lambda v: v)

# ---------------- referência: Naive (random walk) — NÃO entra no ranking dos 4 modelos ----------------
def forecaster_naive(o, j): return np.repeat(y[o], H)

## 7. Modelos e otimização de hiperparâmetros (item 5.5)

Regra: **o teste nunca é usado**. Cada modelo é avaliado por MAE em walk-forward na janela de **validação**
(`ORIGINS_VAL`), treinando somente com dados anteriores a cada origem.

**Tuning menos ruidoso.** Escolher a melhor entre muitas configurações com poucas origens premia a *sorte* (o vencedor da
validação costuma decepcionar no teste). Por isso:
1. a janela de validação é maior e mais densa (`N_VAL`, `STEP_VAL` → ~50 origens);
2. **triagem em 2 etapas**: a etapa 1 avalia *todas* as configurações em 1 de cada `SCREEN_EVERY` origens (barato) e a etapa 2
   reavalia só os `N_KEEP` finalistas na validação **completa**;
3. o MLP, na etapa 2, é avaliado com a **média de `N_SEEDS_TUNE` redes** (menos variância por inicialização);
4. a tabela traz o **erro-padrão do MAE** (entre origens): finalistas separados por menos de ~1 erro-padrão são praticamente empatados.

**Escala.** SARIMAX e Holt-Winters ajustam `log(Close)` e voltam a USD com `exp` — a mesma lógica de RF/MLP, que preveem log-retorno.
O `exp` da previsão do log é a **mediana** condicional, que é justamente o ponto ótimo para o MAE.

In [ ]:
def tuning_duas_etapas(nome, configs, make_fc):
    """etapa 1: todas as configs em origens de validação esparsas; etapa 2: N_KEEP finalistas na validação completa.
    make_fc(cfg, etapa) -> forecaster. Retorna (melhor_cfg, tabela)."""
    orig_s1, s1 = ORIGINS_VAL[::SCREEN_EVERY], []
    for i, cfg in enumerate(configs):
        try:
            d_, t_ = run_walk_forward(make_fc(cfg, 1), orig_s1, verbose=False)
            s1.append((cfg, mae_of(d_), t_)); print(f"  [{nome}] triagem {i+1}/{len(configs)}  MAE={s1[-1][1]:.4f}")
        except Exception as e:
            print(f"  [{nome}] config falhou {cfg}: {str(e)[:60]}")
    assert s1, f"{nome}: nenhuma configuração válida"
    s1.sort(key=lambda r: r[1])
    s2 = []
    for cfg, m1, t1 in s1[:N_KEEP]:
        d_, t_ = run_walk_forward(make_fc(cfg, 2), ORIGINS_VAL, verbose=False)
        s2.append((cfg, m1, mae_of(d_), mae_se(d_), t1 + t_)); print(f"  [{nome}] finalista  MAE_val={s2[-1][2]:.4f} ± {s2[-1][3]:.4f}")
    s2.sort(key=lambda r: r[2])
    linhas = [{**c, "MAE_triagem": m1, "MAE_val": m2, "EP_val": se, "tempo_s": t} for c, m1, m2, se, t in s2]
    linhas += [{**c, "MAE_triagem": m1, "MAE_val": np.nan, "EP_val": np.nan, "tempo_s": t} for c, m1, t in s1[N_KEEP:]]
    return s2[0][0], pd.DataFrame(linhas)

### 7.1 Holt-Winters (referência univariada)
Espaço de busca: tendência {nenhuma, aditiva, aditiva amortecida} × sazonalidade {nenhuma, aditiva, multiplicativa} ×
período sazonal {5, 21}, sobre `log(Close)`. Os **parâmetros de suavização** (α, β, γ, φ) são estimados por máxima
verossimilhança em cada ajuste. Tendência aditiva no log = crescimento exponencial no preço.

In [ ]:
def make_hw_forecaster(cfg):
    def f(o, j):
        mod = ExponentialSmoothing(
            Y_M[:o + 1], trend=cfg["trend"], damped_trend=cfg["damped"],
            seasonal=cfg["seasonal"], seasonal_periods=cfg["m"] if cfg["seasonal"] else None,
            initialization_method="estimated")
        return INV(np.asarray(mod.fit(optimized=True).forecast(H)))
    return f

HW_GRID = []
for trend, damped in [(None, False), ("add", False), ("add", True)]:
    HW_GRID.append(dict(trend=trend, damped=damped, seasonal=None, m=None))
    for seas in ("add", "mul"):
        for m in (5, 21):
            HW_GRID.append(dict(trend=trend, damped=damped, seasonal=seas, m=m))

HW_BEST, df_hw = tuning_duas_etapas("Holt-Winters", HW_GRID, lambda cfg, etapa: make_hw_forecaster(cfg))
print("\nMelhor Holt-Winters:", HW_BEST)
df_hw.head(8)

### 7.2 SARIMAX — seleção automática com **pmdarima** (`auto_arima`)

**Exógenas sem assimetria.** Para a comparação ser justa, o SARIMAX recebe as **mesmas famílias de variáveis externas** que RF e MLP
(`SX_EXOG_MODE = "equalizado"`): retorno de 1 dia (lags 0 e 1), de 5 e de 21 dias de cada exógena — **todas defasadas em `H`**
(para prever `t+k` usa-se o valor de `t+k−H ≤ t`; nenhum valor futuro). Padronizadas só com o passado.

1. `pmdarima.auto_arima` faz a busca **stepwise** (Hyndman & Khandakar) sobre `(p, d, q)(P, D, Q)ₘ` com `m = 5`, escolhendo `d`
   pelo teste KPSS e `D` pelo teste OCSB, ranqueando por **BIC** (AIC também é registrado), em dados **anteriores à validação**.
   Com `return_valid_fits=True` guardamos todos os modelos válidos visitados.
2. Os `N_TOP_SARIMAX` melhores por BIC são confrontados por **MAE em walk-forward na validação completa**; vence o menor MAE
   (a tabela mostra o erro-padrão).
3. O re-treino a cada origem usa `statsmodels.SARIMAX` (o motor que o pmdarima encapsula) com a ordem escolhida.

In [ ]:
# ---- exógenas do SARIMAX: mesmas famílias de RF/MLP, todas defasadas em H ----
SX_SPEC = [("ret1", 0), ("ret1", 1), ("ret5d", 0), ("ret21d", 0)] if SX_EXOG_MODE == "equalizado" else [("ret1", 0)]
def _feat_exog(c, kind): return {"ret1": chg(df[c], 1), "ret5d": chg(df[c], 5), "ret21d": chg(df[c], 21)}[kind]
_cols, EXOG_SX_NAMES = [], []
for c in EXOG_COLS:
    for kind, extra in SX_SPEC:
        _cols.append(_feat_exog(c, kind).shift(H + extra).values)       # valor em t+k-H (extra = lag adicional)
        EXOG_SX_NAMES.append(f"{c} {kind}" + (f" lag{extra}" if extra else "") + f" (defas. {H + extra})")
EXOG_SX = np.column_stack(_cols)
SX_FIRST = int(np.where(~np.isnan(EXOG_SX).any(axis=1))[0][0])
print(f"SARIMAX: {EXOG_SX.shape[1]} regressores exógenos ({SX_EXOG_MODE}) | 1º índice válido: {SX_FIRST}")

def make_sarimax_forecaster(cfg, usar_exog=True):
    st = {"p": None}
    def f(o, j):
        a, kw = SX_FIRST, {}
        if usar_exog:
            ex_tr, ex_fut = EXOG_SX[a:o + 1], EXOG_SX[o + 1:o + H + 1]
            sc = StandardScaler().fit(ex_tr)                               # escala só com o passado
            kw["exog"] = sc.transform(ex_tr)
        mod = SARIMAX(Y_M[a:o + 1], order=cfg["order"], seasonal_order=cfg["sorder"], trend=cfg["trend"],
                      enforce_stationarity=False, enforce_invertibility=False, **kw)
        try:    res = mod.fit(disp=False, maxiter=200, start_params=st["p"])   # começa do ajuste anterior (mais rápido)
        except Exception: res = mod.fit(disp=False, maxiter=200)
        st["p"] = res.params
        fc = res.forecast(steps=H, exog=sc.transform(ex_fut)) if usar_exog else res.forecast(steps=H)
        return INV(np.asarray(fc))
    return f

# ---------------- pmdarima.auto_arima (somente dados anteriores à validação) ----------------
a_, b_ = SX_FIRST, VAL_START
sc_g = StandardScaler().fit(EXOG_SX[a_:b_]); ex_g = sc_g.transform(EXOG_SX[a_:b_])
PM_LIMITES = dict(max_p=3, max_q=3, max_P=2, max_Q=2, max_d=2, max_D=1, maxiter=50)
if QUICK_MODE: PM_LIMITES = dict(max_p=2, max_q=2, max_P=1, max_Q=1, max_d=2, max_D=1, maxiter=20)

t0 = time.time()
fits_pm = pm.auto_arima(
    Y_M[a_:b_], X=ex_g,
    start_p=1, start_q=1, start_P=0, start_Q=0,
    seasonal=True, m=SEASONAL_M,
    d=None, D=None, test="kpss", seasonal_test="ocsb",          # d e D escolhidos por testes de raiz unitária
    information_criterion="bic", stepwise=True, with_intercept="auto",
    suppress_warnings=True, error_action="ignore", trace=1, n_jobs=1,
    return_valid_fits=True, **PM_LIMITES)
print(f"\nauto_arima: {len(fits_pm)} modelos válidos em {time.time()-t0:.0f}s")

def cfg_from_pm(m_):
    order = tuple(int(v) for v in m_.order); so = tuple(int(v) for v in m_.seasonal_order)
    if so[:3] == (0, 0, 0): so = (0, 0, 0, 0)                                   # sem parte sazonal
    return dict(order=order, sorder=so, trend="c" if (order[1] == 0 and so[1] == 0) else None)

linhas, vistos = [], set()
for m_ in fits_pm:
    cfg = cfg_from_pm(m_); chave = (cfg["order"], cfg["sorder"], cfg["trend"])
    if chave in vistos: continue
    vistos.add(chave); linhas.append({**cfg, "AIC": m_.arima_res_.aic, "BIC": m_.arima_res_.bic})
df_sx_ic = pd.DataFrame(linhas).sort_values("BIC").reset_index(drop=True)
print("Melhores modelos visitados pelo auto_arima (ranking por BIC, dados pré-validação):")
print(df_sx_ic.head(8).round(1).to_string(index=False))

# confronta os melhores por BIC em walk-forward de validação COMPLETA
cands = []
for _, r in df_sx_ic.head(N_TOP_SARIMAX).iterrows():
    cfg = dict(order=r["order"], sorder=r["sorder"], trend=r["trend"])
    d_, t_ = run_walk_forward(make_sarimax_forecaster(cfg), ORIGINS_VAL, verbose=False)
    cands.append({**cfg, "BIC": r["BIC"], "AIC": r["AIC"], "MAE_val": mae_of(d_), "EP_val": mae_se(d_), "tempo_s": t_})
df_sx = pd.DataFrame(cands).sort_values("MAE_val").reset_index(drop=True)
SX_BEST = {k: df_sx.iloc[0][k] for k in ("order", "sorder", "trend")}
print("\nMelhor SARIMAX (menor MAE de validação):", SX_BEST)
df_sx

### 7.3 Random Forest
Espaço: `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`. Método: **busca aleatória**
(`N_ITER_RF` configurações) com **triagem em 2 etapas**. A floresta é multi-saída (uma árvore prevê os H alvos).

In [ ]:
def make_tab_forecaster(fit_fn, predict_fn, refit_every=1, cols=None):
    """cols: índices das features a usar (None = todas). Treino só com linhas cujo alvo já foi observado na origem."""
    Fa = F_arr if cols is None else F_arr[:, cols]
    st = {"model": None, "last": None}
    def f(o, j):
        if st["model"] is None or (j - st["last"]) >= refit_every:
            idx = train_rows(o)
            st["model"], st["last"] = fit_fn(Fa[idx], Y[idx]), j
        pred_t = predict_fn(st["model"], Fa[o:o + 1])[0]
        return back_transf(y[o], pred_t)
    return f

def fit_rf(X, Yt, cfg): return RandomForestRegressor(random_state=SEED, n_jobs=-1, **cfg).fit(X, Yt)
def pred_rf(m, X):      return m.predict(X)

RF_SPACE = dict(n_estimators=[100, 200, 400], max_depth=[3, 5, 8, 12, None], min_samples_split=[2, 5, 10, 20],
                min_samples_leaf=[1, 3, 5, 10], max_features=[0.3, 0.5, 0.8, 1.0, "sqrt"])

def mk_rf(cfg, etapa): return make_tab_forecaster(lambda X, Yt, c=cfg: fit_rf(X, Yt, c), pred_rf)
RF_BEST, df_rf = tuning_duas_etapas("Random Forest", amostrar(RF_SPACE, N_ITER_RF), mk_rf)
print("\nMelhor Random Forest:", RF_BEST)
df_rf.head(6)

### 7.4 MLP Regressor (PyTorch) — modelo de especialização

Arquitetura: `Linear → ReLU → (Dropout) → … → Linear(H)` — saída linear de **H neurônios** (um por horizonte).
Otimizador: Adam. **Early stopping manual** sobre os últimos 15% cronológicos do treino (equivalente ao
`EarlyStopping(monitor="val_loss", restore_best_weights=True)` do Keras da aula). Entradas e alvos são **padronizados**
(`StandardScaler` ajustado só no treino de cada origem).

**Perda ponderada por horizonte.** Ao padronizar cada saída pelo seu desvio-padrão, todos os horizontes passam a pesar igual na loss.
Mas o MAE em USD pesa mais os horizontes longos (erros maiores). Com `peso_horizonte=True` a loss vira
`mean(wₖ · |ŷₖ − yₖ|)` com `wₖ ∝ σₖ` — que é exatamente o **L1 na escala original**, alinhado com a métrica do trabalho. Com `False`
fica o L1 padronizado uniforme. Essa escolha **entra no espaço de busca** (a validação decide).

**Média de redes.** O resultado final é a média de `N_SEEDS_FINAL` redes treinadas com sementes diferentes (reduz a variância
de inicialização/early stopping).

In [ ]:
class MLPNet(nn.Module):
    def __init__(self, n_in, n_out, hidden=(64, 64), dropout=0.0):
        super().__init__()
        layers, d = [], n_in
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU()]
            if dropout > 0: layers.append(nn.Dropout(dropout))
            d = h
        layers.append(nn.Linear(d, n_out))            # saída linear (regressão)
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def treinar_mlp(Xs, Ys, cfg, max_epochs, patience, seed, w, val_frac=0.15):
    """w: pesos por horizonte (H,). Loss = mean(w * |pred - y|) -> L1 uniforme se w = 1."""
    torch.manual_seed(seed); np.random.seed(seed)
    n_val = max(int(len(Xs) * val_frac), 30)
    T = lambda a: torch.tensor(a, dtype=torch.float32, device=device)
    Xtr, Ytr, Xva, Yva, wt = T(Xs[:-n_val]), T(Ys[:-n_val]), T(Xs[-n_val:]), T(Ys[-n_val:]), T(w)
    net = MLPNet(Xs.shape[1], Ys.shape[1], cfg["hidden"], cfg["dropout"]).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    loss_fn = lambda p, t: ((p - t).abs() * wt).mean()
    n_tr = Xtr.shape[0]
    best, best_state, wait, hist = float("inf"), None, 0, {"loss": [], "val_loss": []}
    for ep in range(max_epochs):
        net.train(); perm = torch.randperm(n_tr).to(device); tot = 0.0
        for i in range(0, n_tr, cfg["batch_size"]):
            b = perm[i:i + cfg["batch_size"]]
            opt.zero_grad(); loss = loss_fn(net(Xtr[b]), Ytr[b]); loss.backward(); opt.step()
            tot += loss.item() * len(b)
        net.eval()
        with torch.no_grad(): vl = loss_fn(net(Xva), Yva).item()
        hist["loss"].append(tot / n_tr); hist["val_loss"].append(vl)
        if vl < best - 1e-6:
            best, wait = vl, 0
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break                  # early stopping
    net.load_state_dict(best_state); net.eval()         # restore_best_weights
    return net, hist

class MLPEnsemble:
    """média das previsões de várias redes (sementes diferentes)"""
    def __init__(self, nets, sx, sy, hist): self.nets, self.sx, self.sy, self.hist = nets, sx, sy, hist
    def predict(self, X):
        Xs = torch.tensor(self.sx.transform(X), dtype=torch.float32, device=device)
        with torch.no_grad(): outs = [n.eval()(Xs).cpu().numpy() for n in self.nets]
        return self.sy.inverse_transform(np.mean(outs, axis=0))

def fit_mlp(X, Yt, cfg, epochs, n_seeds=1):
    sx, sy = StandardScaler().fit(X), StandardScaler().fit(Yt)
    Xs, Ys = sx.transform(X), sy.transform(Yt)
    # w_k ∝ desvio do horizonte k  =>  loss padronizada ponderada == L1 na escala original
    w = sy.scale_ / sy.scale_.mean() if cfg.get("peso_horizonte", True) else np.ones(Yt.shape[1])
    fits = [treinar_mlp(Xs, Ys, cfg, epochs, MLP_PATIENCE, SEED + s, w) for s in range(n_seeds)]
    return MLPEnsemble([f[0] for f in fits], sx, sy, fits[0][1])
def pred_mlp(m, X): return m.predict(X)

MLP_SPACE = dict(hidden=[(32,), (64,), (64, 64), (128, 64), (128, 64, 32)], dropout=[0.0, 0.1, 0.2, 0.3],
                 lr=[3e-4, 1e-3, 3e-3], weight_decay=[0.0, 1e-5, 1e-4, 1e-3], batch_size=[32, 64, 128],
                 peso_horizonte=[True, False])

In [ ]:
def mk_mlp(cfg, etapa):
    ns = 1 if etapa == 1 else N_SEEDS_TUNE          # etapa 2: média de redes -> menos ruído
    return make_tab_forecaster(lambda X, Yt, c=cfg, n=ns: fit_mlp(X, Yt, c, MLP_EPOCHS_TUNE, n), pred_mlp)

MLP_BEST, df_mlp = tuning_duas_etapas("MLP", amostrar(MLP_SPACE, N_ITER_MLP), mk_mlp)
print("\nMelhor MLP:", MLP_BEST)
df_mlp.head(6)

### 7.5 Resumo dos hiperparâmetros escolhidos (vai para a seção 7 do relatório)

In [ ]:
def _clean(v):
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating,)): return float(v)
    if isinstance(v, np.bool_): return bool(v)
    if isinstance(v, tuple): return [_clean(i) for i in v]
    return v
BEST = {"SARIMAX": SX_BEST, "Holt-Winters": HW_BEST, "Random Forest": RF_BEST, "MLP (PyTorch)": MLP_BEST}
BEST = {m: {k: _clean(v) for k, v in c.items()} for m, c in BEST.items()}
metodo = {"SARIMAX": f"pmdarima.auto_arima (stepwise, BIC, d/D por KPSS/OCSB) sobre {'log(Close)' if MODEL_LOG else 'Close'} + exógenas {SX_EXOG_MODE}s defasadas em H; top-{N_TOP_SARIMAX} por BIC confrontados por MAE de validação",
          "Holt-Winters": f"grade completa (tendência × amortecimento × sazonalidade × m) sobre {'log(Close)' if MODEL_LOG else 'Close'}; triagem em 2 etapas",
          "Random Forest": f"busca aleatória ({N_ITER_RF} configs); triagem em 2 etapas",
          "MLP (PyTorch)": f"busca aleatória ({N_ITER_MLP} configs, inclui peso por horizonte); triagem em 2 etapas; média de {N_SEEDS_FINAL} redes no final"}
pd.DataFrame({"modelo": list(BEST), "método de busca": [metodo[m] for m in BEST], "configuração final": [str(BEST[m]) for m in BEST]})

## 8. Treinamento e previsão no TESTE (item 5.6)

Hiperparâmetros **fixos**; cada modelo é re-treinado a cada origem só com dados disponíveis naquele momento.
São registrados: parâmetros, tempo de execução, previsões e resíduos. O **Naive** (último valor) entra apenas como
referência — em séries de preço de ações é um adversário difícil.

> Tempo: o MLP treina `N_SEEDS_FINAL` redes por origem. Se estiver lento, reduza `N_SEEDS_FINAL` ou aumente `REFIT_EVERY_TAB`
> (ex.: 4 re-treina a cada ~4 origens) — e registre a escolha no relatório.

In [ ]:
PRED, TEMPO = {}, {}
PRED_ABL, TEMPO_ABL = {}, {}

def rodar(nome, forecaster, store=PRED, tstore=TEMPO):
    p = f"{RESULTS_DIR}/pred_{BASE_NAME}_{re.sub(r'[^0-9A-Za-z]+', '_', nome)}.csv"
    if USE_CACHE and os.path.exists(p):
        store[nome] = pd.read_csv(p, parse_dates=["origem", "data"]); tstore[nome] = json.load(open(p + ".json"))["tempo_s"]
        print(f"{nome}: carregado do cache"); return
    print(f"Rodando {nome} ...")
    store[nome], tstore[nome] = run_walk_forward(forecaster, ORIGINS_TEST, nome)
    store[nome].to_csv(p, index=False); json.dump({"tempo_s": tstore[nome]}, open(p + ".json", "w"))
    print(f"   MAE teste = {mae_of(store[nome]):.4f} | tempo = {tstore[nome]:.0f}s")

rodar("Naive (referência)", forecaster_naive)
rodar("Holt-Winters", make_hw_forecaster(HW_BEST))
rodar("SARIMAX", make_sarimax_forecaster(SX_BEST))
rodar("Random Forest", make_tab_forecaster(lambda X, Yt: fit_rf(X, Yt, RF_BEST), pred_rf, REFIT_EVERY_TAB))
rodar("MLP (PyTorch)", make_tab_forecaster(lambda X, Yt: fit_mlp(X, Yt, MLP_BEST, MLP_EPOCHS_FINAL, N_SEEDS_FINAL), pred_mlp, REFIT_EVERY_TAB))

## 9. Comparação por MAE (item 6)

O MAE é calculado sobre **todas** as previsões fora da amostra do walk-forward (todas as origens × horizontes 1…H), em USD.
Não se somam/médiam MAEs entre bases (escalas diferentes) — a comparação entre bases usa **posição e vitórias** (seção 13).

In [ ]:
MODELOS = ["SARIMAX", "Holt-Winters", "Random Forest", "MLP (PyTorch)"]
linhas = []
for nome, d in PRED.items():
    ae = d["erro"].abs()
    linhas.append({"modelo": nome, "MAE": ae.mean(), "EP_MAE": mae_se(d), "RMSE": np.sqrt((d["erro"] ** 2).mean()),
                   "MAPE_%": (ae / d["real"]).mean() * 100, "tempo_s": TEMPO[nome],
                   **{f"MAE_h{k}": ae[d["passo"] == k].mean() for k in range(1, H + 1)}})
tab = pd.DataFrame(linhas).set_index("modelo")
naive_mae = tab.loc["Naive (referência)", "MAE"]
tab["MAE/Naive"] = tab["MAE"] / naive_mae

rank = tab.loc[MODELOS].sort_values("MAE").copy(); rank.insert(0, "posição", range(1, len(rank) + 1))
print(f"Base {BASE_NAME} | teste: {dates[TEST_START].date()} -> {dates[-1].date()} | {len(ORIGINS_TEST)} origens × H={H}")
print("Referência Naive (último valor): MAE =", round(naive_mae, 4))
print("MELHOR MODELO nesta base (por MAE):", rank.index[0], "— veja a seção 9.1: a diferença é estatisticamente real?")
print("\nRANKING (4 modelos do trabalho, do menor para o maior MAE):")
rank[["posição", "MAE", "EP_MAE", "RMSE", "MAPE_%", "MAE/Naive", "tempo_s"]].round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
real = PRED["SARIMAX"].sort_values(["data", "passo"])
ax[0].plot(real["data"], real["real"], color="black", lw=1.5, label="Real")
for m, c in zip(MODELOS, ["tab:blue", "tab:orange", "tab:green", "tab:red"]):
    d = PRED[m].sort_values(["data", "passo"]); ax[0].plot(d["data"], d["pred"], lw=0.9, alpha=0.85, color=c, label=m)
ax[0].set_title("Teste: real vs. previsto (walk-forward)"); ax[0].legend(fontsize=8)
w = 0.2
for i, m in enumerate(MODELOS + ["Naive (referência)"]):
    ax[1].bar(np.arange(1, H + 1) + (i - 2) * w, [tab.loc[m, f"MAE_h{k}"] for k in range(1, H + 1)], width=w, label=m)
ax[1].set_xlabel("passo à frente (h)"); ax[1].set_ylabel("MAE (USD)"); ax[1].set_title("MAE por horizonte"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

### 9.1 As diferenças de MAE são reais? (significância estatística)

Com ~100 origens, dois MAEs próximos podem ser só ruído. Comparamos cada par de modelos (e cada modelo contra o Naive) com:

- **Teste de Diebold-Mariano** sobre o **MAE por origem** (uma perda por origem; evita contar 5 vezes erros do mesmo bloco),
  com variância robusta **Newey-West** (defasagem curta, ≈ ½·n^(1/3)) e correção de **Harvey-Leybourne-Newbold** (amostra pequena), p-valor bilateral.
  Em simulação com n≈100 o teste tem taxa de falso positivo ≈ 6% (série iid) a ≈ 9% (autocorrelação moderada) — levemente liberal; por isso a conclusão exige **também** p de Holm < 0,05 **e** IC do bootstrap sem o zero.
- **Bootstrap por blocos** (`BOOT_B` repetições) da diferença média de MAE → **IC de 95%**.
- **Correção de Holm** dos p-valores, pois são vários testes simultâneos (evita "achar" diferença por acaso).

Leitura: `ΔMAE (A−B) < 0` ⇒ A melhor. Só vale como diferença real se o p-valor ajustado < 0,05 **e** o IC não contiver 0.

In [ ]:
def loss_origem(d): return mae_por_origem(d).values            # um MAE por origem (mesma ordem para todos os modelos)
LOSS = {m: loss_origem(d) for m, d in PRED.items()}

def dm_test(la, lb):
    """Diebold-Mariano (HAC Newey-West + correção HLN, h=1 sobre perdas agregadas por origem). H0: acurácia igual."""
    d = la - lb; n = len(d); dbar = d.mean()
    q = max(1, int(round(0.5 * n ** (1 / 3))))          # defasagem Newey-West curta (calibrada por simulação p/ n≈60–100)
    g = [np.sum((d[k:] - dbar) * (d[:n - k] - dbar)) / n for k in range(q + 1)]
    var = max(g[0] + 2 * sum((1 - k / (q + 1)) * g[k] for k in range(1, q + 1)), 1e-12)
    stat = dbar / np.sqrt(var / n) * np.sqrt((n - 1) / n)
    return dbar, stat, 2 * stats.t.sf(abs(stat), df=n - 1)

def boot_ic(d, B=BOOT_B, seed=SEED):
    """IC 95% da média de d por bootstrap em blocos móveis (preserva autocorrelação)."""
    rng = np.random.default_rng(seed); n = len(d); b = max(2, int(round(n ** (1 / 3)))); nb = int(np.ceil(n / b))
    starts = rng.integers(0, n - b + 1, size=(B, nb))
    idx = (starts[:, :, None] + np.arange(b)).reshape(B, -1)[:, :n]
    return tuple(np.percentile(d[idx].mean(axis=1), [2.5, 97.5]))

def holm(p):
    p = np.asarray(p); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o): run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj

NAIVE = "Naive (referência)"
pares = list(itertools.combinations(MODELOS, 2)) + [(m, NAIVE) for m in MODELOS]
linhas = []
for A, B in pares:
    dif = LOSS[A] - LOSS[B]; dbar, stat, p = dm_test(LOSS[A], LOSS[B]); lo, hi = boot_ic(dif)
    linhas.append({"A": A, "B": B, "MAE_A": LOSS[A].mean(), "MAE_B": LOSS[B].mean(), "ΔMAE (A−B)": dbar, "IC95 inf": lo, "IC95 sup": hi, "p (DM)": p})
tab_dm = pd.DataFrame(linhas); tab_dm["p (Holm)"] = holm(tab_dm["p (DM)"])
def _leitura(r):
    if r["p (Holm)"] < 0.05 and not (r["IC95 inf"] <= 0 <= r["IC95 sup"]): return f"{r['A'] if r['ΔMAE (A−B)'] < 0 else r['B']} é melhor (significativo)"
    return "indistinguíveis (diferença pode ser ruído)"
tab_dm["leitura"] = tab_dm.apply(_leitura, axis=1)

fig, ax = plt.subplots(figsize=(9, 0.5 * len(tab_dm) + 1.5))
for i, r in tab_dm.iterrows():
    ax.errorbar(r["ΔMAE (A−B)"], i, xerr=[[r["ΔMAE (A−B)"] - r["IC95 inf"]], [r["IC95 sup"] - r["ΔMAE (A−B)"]]], fmt="o", capsize=3,
                color="tab:red" if r["p (Holm)"] < 0.05 else "tab:gray")
ax.axvline(0, color="black", lw=1); ax.set_yticks(range(len(tab_dm))); ax.set_yticklabels([f"{a}  vs  {b}" for a, b in zip(tab_dm["A"], tab_dm["B"])], fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("ΔMAE (A − B) em USD · negativo = A melhor · vermelho = significativo (Holm 5%)"); ax.set_title("Diferença de MAE com IC 95% (bootstrap por blocos)")
plt.tight_layout(); plt.show()
tab_dm.round(4)

### 9.2 Ablação: as variáveis externas realmente ajudam?

A importância por permutação mostra **o que o modelo usa**, não se isso **melhora a previsão**. Aqui rodamos RF, MLP e SARIMAX
**sem** as variáveis externas (mantendo histórico do alvo, OHLCV e calendário) e comparamos o MAE **com × sem** exógenas,
com o mesmo teste de Diebold-Mariano e bootstrap. Usam-se os **mesmos hiperparâmetros** (sem re-otimizar) para isolar o efeito
das exógenas — uma re-otimização separada poderia mudar levemente o resultado. O Holt-Winters é univariado (não se aplica).

In [ ]:
IDX_SEM_EXOG = [i for i, f in enumerate(FEATURES) if GRUPO[f] != "externa"]
print(f"features: {len(FEATURES)} (com) -> {len(IDX_SEM_EXOG)} (sem exógenas)")

if RODAR_ABLACAO:
    rodar("SARIMAX | sem exógenas", make_sarimax_forecaster(SX_BEST, usar_exog=False), PRED_ABL, TEMPO_ABL)
    rodar("Random Forest | sem exógenas",
          make_tab_forecaster(lambda X, Yt: fit_rf(X, Yt, RF_BEST), pred_rf, REFIT_EVERY_TAB, cols=IDX_SEM_EXOG), PRED_ABL, TEMPO_ABL)
    rodar("MLP (PyTorch) | sem exógenas",
          make_tab_forecaster(lambda X, Yt: fit_mlp(X, Yt, MLP_BEST, MLP_EPOCHS_FINAL, N_SEEDS_FINAL), pred_mlp, REFIT_EVERY_TAB, cols=IDX_SEM_EXOG),
          PRED_ABL, TEMPO_ABL)

linhas = []
for m in ["SARIMAX", "Random Forest", "MLP (PyTorch)"]:
    if f"{m} | sem exógenas" not in PRED_ABL: continue
    lc, ls = LOSS[m], loss_origem(PRED_ABL[f"{m} | sem exógenas"]); dif = lc - ls
    dbar, stat, p = dm_test(lc, ls); lo, hi = boot_ic(dif)
    if p < 0.05 and not (lo <= 0 <= hi): leit = "exógenas AJUDAM (significativo)" if dbar < 0 else "exógenas PIORAM (significativo)"
    else: leit = "sem efeito detectável das exógenas"
    linhas.append({"modelo": m, "MAE com exóg.": lc.mean(), "MAE sem exóg.": ls.mean(), "Δ (com−sem)": dbar,
                   "Δ %": dbar / ls.mean() * 100, "IC95 inf": lo, "IC95 sup": hi, "p (DM)": p, "leitura": leit})
tab_abl = pd.DataFrame(linhas)
tab_abl.round(4) if len(tab_abl) else print("ablação desativada (RODAR_ABLACAO=False)")

## 10. Análise dos resíduos (item 7)

Resíduo = `real − previsto`, fora da amostra, na ordem cronológica. Para cada modelo: gráfico no tempo, **ACF** e
**teste de Ljung-Box** (H0: os resíduos são ruído branco → p < 0,05 indica autocorrelação remanescente).
Também reportamos o **viés** (média dos resíduos, com teste t) e a variabilidade.

> Observação metodológica: como cada origem prevê H dias, os erros de um mesmo bloco são naturalmente correlacionados
> (o erro de `t+3` contém o de `t+2`). Por isso reportamos Ljung-Box também na série do **passo 1**, que não sofre desse efeito.

In [ ]:
linhas = []
fig, axes = plt.subplots(len(MODELOS), 2, figsize=(13, 2.7 * len(MODELOS)))
for i, m in enumerate(MODELOS):
    d = PRED[m].sort_values(["data", "passo"]); e = d["erro"].values
    axes[i, 0].plot(d["data"], e, lw=0.6); axes[i, 0].axhline(0, color="red", ls="--", lw=1)
    axes[i, 0].set_title(f"{m} — resíduos no tempo (USD)", fontsize=9)
    plot_acf(e, lags=30, ax=axes[i, 1], title=f"{m} — ACF dos resíduos", zero=False)
    lb  = acorr_ljungbox(e, lags=[5, 10, 20], return_df=True)
    lb1 = acorr_ljungbox(d.loc[d["passo"] == 1, "erro"].values, lags=[10], return_df=True)
    t, p_t = stats.ttest_1samp(e, 0.0)
    linhas.append({"modelo": m, "viés (média)": e.mean(), "desvio": e.std(ddof=1), "p (viés≠0)": p_t,
                   "LB p (lag 5)": lb.loc[5, "lb_pvalue"], "LB p (lag 10)": lb.loc[10, "lb_pvalue"], "LB p (lag 20)": lb.loc[20, "lb_pvalue"],
                   "LB p passo 1 (lag 10)": lb1.loc[10, "lb_pvalue"]})
plt.tight_layout(); plt.show()
tab_lb = pd.DataFrame(linhas).set_index("modelo")
tab_lb.round(4)

In [ ]:
# leitura automática (a interpretação final, com contexto, é do relatório)
for m, r in tab_lb.iterrows():
    v = "com viés significativo" if r["p (viés≠0)"] < 0.05 else "sem viés significativo"
    a = "há autocorrelação remanescente" if r["LB p (lag 10)"] < 0.05 else "sem evidência de autocorrelação"
    a1 = "autocorrelação no passo 1" if r["LB p passo 1 (lag 10)"] < 0.05 else "passo 1 sem autocorrelação"
    print(f"{m:<15} viés={r['viés (média)']:+.3f} USD ({v}) | Ljung-Box (série completa, lag 10): {a} | {a1}")

**O que discutir no relatório:** (i) viés — os modelos subestimam/superestimam sistematicamente? (ii) picos de resíduo
coincidem com choques (resultados trimestrais)? (iii) ACF/Ljung-Box com p < 0,05 significam que ainda há estrutura linear
que o modelo não capturou; (iv) variabilidade dos resíduos varia no tempo (heterocedasticidade, típica de finanças)?

## 10.1 Análise por regime (onde cada modelo acerta ou erra)

Um MAE médio esconde *quando* o modelo falha. Quebramos o erro por **ano**, por **regime de volatilidade** (volatilidade de 21 dias
na origem, em tercis calculados só com dados **pré-teste**) e por **regime de tendência** (retorno de 63 dias na origem: queda < −5% <
lateral < +5% < alta). Também medimos a **acurácia direcional**: o modelo acertou se o preço ia subir ou cair em relação ao
fechamento da origem? (O Naive não tem direção — prevê "nada muda".)

In [ ]:
TODOS = MODELOS + ["Naive (referência)"]
vol_pre = F["vol21d"].values[VALID & (ROW < TEST_START)]
VQ1, VQ2 = np.quantile(vol_pre, [1 / 3, 2 / 3])                 # tercis de volatilidade SÓ do período pré-teste
print(f"Tercis de volatilidade (21d, pré-teste): {VQ1:.4f} | {VQ2:.4f}")

def com_regimes(d):
    d = d.copy(); ori = pd.DatetimeIndex(d["origem"])
    d["ano"] = d["data"].dt.year
    d["reg_vol"] = pd.cut(F["vol21d"].reindex(ori).values, [-np.inf, VQ1, VQ2, np.inf], labels=["vol. baixa", "vol. média", "vol. alta"])
    d["reg_tend"] = pd.cut(F["ret63d"].reindex(ori).values, [-np.inf, -0.05, 0.05, np.inf], labels=["queda (63d)", "lateral (63d)", "alta (63d)"])
    d["ae"] = d["erro"].abs()
    return d
DR = {m: com_regimes(PRED[m]) for m in TODOS}

def mae_por(chave):
    t = pd.DataFrame({m: DR[m].groupby(chave, observed=True)["ae"].mean() for m in TODOS})
    t.insert(0, "origens", DR[NAIVE].groupby(chave, observed=True)["origem"].nunique())
    return t

tab_ano, tab_vol, tab_tend = mae_por("ano"), mae_por("reg_vol"), mae_por("reg_tend")
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for a_, t_, tit in zip(ax, (tab_ano, tab_vol, tab_tend), ("MAE por ano", "MAE por regime de volatilidade", "MAE por regime de tendência")):
    t_[TODOS].plot.bar(ax=a_, width=0.85, rot=0); a_.set_title(tit); a_.set_ylabel("MAE (USD)"); a_.legend(fontsize=6)
plt.tight_layout(); plt.show()
print("MAE por ano (origens = nº de origens de teste no grupo):"); display(tab_ano.round(3)) if "display" in globals() else print(tab_ano.round(3))
print("MAE por regime de volatilidade:"); display(tab_vol.round(3)) if "display" in globals() else print(tab_vol.round(3))
print("MAE por regime de tendência:"); display(tab_tend.round(3)) if "display" in globals() else print(tab_tend.round(3))

In [ ]:
# ---- acurácia direcional: acertou se subiu/caiu em relação ao fechamento da origem? ----
base_d = PRED["SARIMAX"]; sobe_H = (base_d.loc[base_d["passo"] == H, "real"] > base_d.loc[base_d["passo"] == H, "close_origem"]).mean()
linhas = []
for m in MODELOS:
    d = PRED[m]; hit = np.sign(d["pred"] - d["close_origem"]) == np.sign(d["real"] - d["close_origem"])
    hH = hit[d["passo"] == H]; k, n = int(hH.sum()), len(hH)               # h=H: uma observação por origem (independentes)
    linhas.append({"modelo": m, "acerto (todos os h)": hit.mean(), f"acerto h={H}": hH.mean(),
                   "p (binomial vs 50%)": stats.binomtest(k, n, 0.5).pvalue, "p (vs 'sempre alta')": stats.binomtest(k, n, sobe_H).pvalue})
tab_dir = pd.DataFrame(linhas).set_index("modelo")
print(f"Referências em h={H}: chute aleatório = 50% | 'sempre prever alta' = {sobe_H:.1%} (fração de vezes que o preço subiu no teste)")
tab_dir.round(4)

**Como ler:** se o MAE de um modelo é bom só em um ano/regime (ex.: volatilidade baixa), ele não é robusto. Acurácia direcional só
é informativa se **superar o "sempre alta"** (que já acerta mais de 50% se o preço teve deriva positiva) — o teste `p (vs 'sempre alta')` mede isso.

## 11. Importância das features (item 8)

| Modelo | Método |
|---|---|
| SARIMAX | coeficientes das exógenas (sinal, magnitude, significância) |
| Holt-Winters | sem *feature importance* — interpreta-se nível, tendência e sazonalidade |
| Random Forest | importância nativa (impureza) **e** Permutation Importance |
| MLP | Permutation Importance (aumento do MAE ao embaralhar cada feature) |

Para RF/MLP usa-se o modelo treinado **até o início do teste** e a permutação é medida nas linhas do teste
(interpretação, não seleção). As variáveis **externas** são destacadas em vermelho.

In [ ]:
# ---------- SARIMAX: coeficientes (modelo ajustado com todos os dados até a última origem) ----------
o_last = ORIGINS_TEST[-1]
ex_tr = EXOG_SX[SX_FIRST:o_last + 1]; sc_f = StandardScaler().fit(ex_tr)
fit_sx = SARIMAX(Y_M[SX_FIRST:o_last + 1], exog=sc_f.transform(ex_tr), order=SX_BEST["order"], seasonal_order=SX_BEST["sorder"],
                 trend=SX_BEST["trend"], enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)
ren = {f"x{i+1}": n for i, n in enumerate(EXOG_SX_NAMES)}
tab_sx = pd.DataFrame({"coeficiente": np.asarray(fit_sx.params), "erro_padrão": np.asarray(fit_sx.bse),
                       "z": np.asarray(fit_sx.tvalues), "p_valor": np.asarray(fit_sx.pvalues)},
                      index=[ren.get(n, n) for n in fit_sx.model.param_names])
tab_sx["sinal"] = np.sign(tab_sx["coeficiente"]).map({1.0: "+", -1.0: "−", 0.0: "0"})
tab_sx["significativo (5%)"] = tab_sx["p_valor"] < 0.05
print("Coeficientes por 1 desvio-padrão da variável (padronizada), em " + ("log-pontos de Close (≈ variação % no preço)." if MODEL_LOG else "USD."))
print(f"Regressores significativos a 5%: {int(tab_sx['significativo (5%)'].sum())} de {len(EXOG_SX_NAMES)}")
tab_sx.round(4)

In [ ]:
# ---------- Holt-Winters: nível, tendência e sazonalidade ----------
fit_hw = ExponentialSmoothing(Y_M[:o_last + 1], trend=HW_BEST["trend"], damped_trend=HW_BEST["damped"],
                              seasonal=HW_BEST["seasonal"], seasonal_periods=HW_BEST["m"] if HW_BEST["seasonal"] else None,
                              initialization_method="estimated").fit(optimized=True)
par = {k: v for k, v in fit_hw.params.items() if not k.startswith(("initial_seasons",))}
print("Configuração:", HW_BEST)
print(pd.Series(par).round(4).to_string())
for nome_attr in ("level", "trend", "season"):
    v = getattr(fit_hw, nome_attr, None)
    if v is not None and len(np.atleast_1d(v)) > 0: print(f"último valor de {nome_attr}: {np.atleast_1d(v)[-1]:.4f}")
print("\nHolt-Winters não usa variáveis externas: interpretar α (nível), β (tendência), γ (sazonal) e φ (amortecimento).")

In [ ]:
# ---------- RF e MLP: modelos treinados só com dados pré-teste; importância medida nas linhas do teste ----------
o_pre = TEST_START - 1
idx_pre = train_rows(o_pre)
rf_pre  = fit_rf(F_arr[idx_pre], Y[idx_pre], RF_BEST)
mlp_pre = fit_mlp(F_arr[idx_pre], Y[idx_pre], MLP_BEST, MLP_EPOCHS_FINAL, N_SEEDS_FINAL)
X_te, Y_te = F_arr[ORIGINS_TEST], Y[ORIGINS_TEST]

def perm_importance(predict_fn, X, Yt, nomes, n_repeats=10, seed=SEED):
    rng = np.random.default_rng(seed)
    base = mean_absolute_error(Yt, predict_fn(X)); out = []
    for j, nome in enumerate(nomes):
        inc = []
        for _ in range(n_repeats):
            Xp = X.copy(); Xp[:, j] = rng.permutation(Xp[:, j])
            inc.append(mean_absolute_error(Yt, predict_fn(Xp)) - base)
        out.append((nome, np.mean(inc), np.std(inc)))
    r = pd.DataFrame(out, columns=["feature", "aumento_MAE", "desvio"]); r["grupo"] = r["feature"].map(GRUPO)
    return r.sort_values("aumento_MAE", ascending=False).reset_index(drop=True)

imp_rf_nat  = pd.DataFrame({"feature": FEATURES, "importancia": rf_pre.feature_importances_}).assign(grupo=lambda d: d["feature"].map(GRUPO)).sort_values("importancia", ascending=False).reset_index(drop=True)
imp_rf_perm = perm_importance(rf_pre.predict, X_te, Y_te, FEATURES)
imp_mlp     = perm_importance(mlp_pre.predict, X_te, Y_te, FEATURES)

CORES = {"externa": "tab:red", "histórico do alvo": "tab:blue", "própria (OHLCV)": "tab:green", "calendário": "tab:gray"}
def barras(ax, d, col, titulo, top=15):
    dd = d.head(top).iloc[::-1]
    ax.barh(dd["feature"], dd[col], color=[CORES[g] for g in dd["grupo"]]); ax.set_title(titulo, fontsize=10)
fig, ax = plt.subplots(1, 3, figsize=(17, 5.5))
barras(ax[0], imp_rf_nat,  "importancia",  "Random Forest — importância nativa")
barras(ax[1], imp_rf_perm, "aumento_MAE",  "Random Forest — Permutation Importance")
barras(ax[2], imp_mlp,     "aumento_MAE",  "MLP — Permutation Importance")
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=c, label=g) for g, c in CORES.items()], loc="lower center", ncol=4)
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.show()

grp = pd.DataFrame({"RF (permutação)": imp_rf_perm.groupby("grupo")["aumento_MAE"].sum(),
                    "MLP (permutação)": imp_mlp.groupby("grupo")["aumento_MAE"].sum()})
print("Importância agregada por grupo (soma do aumento de MAE, em unidades de log-retorno):"); grp.round(5)

In [ ]:
# curva de aprendizado do MLP pré-teste (diagnóstico de over/underfitting)
h = mlp_pre.hist
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(h["loss"], label="treino (L1)"); ax.plot(h["val_loss"], label="validação (L1)")
ax.set_xlabel("época"); ax.set_ylabel("MAE padronizado"); ax.set_title(f"MLP — curva de aprendizado (parou na época {len(h['loss'])})"); ax.legend()
plt.tight_layout(); plt.show()

**Lembrete de disponibilidade (retomada exigida no item 8):** as exógenas em destaque só entram porque suas observações
**até a origem** já existem no momento da previsão; nenhum valor de `t+1…t+H` é usado. No SARIMAX elas entram defasadas em `H`.
Se uma exógena aparecer como importante, comente **por que faz sentido econômico** (ex.: TSN como concorrente direta, XLP como
proxy de mercado defensivo, USD_MXN pela exposição da PPC ao México) e **por que não é vazamento**.

## 12. Estudo do modelo de especialização: MLP Regressor (item 9)

**Intuição.** Um MLP é uma composição de transformações simples: cada camada oculta calcula combinações lineares das entradas
($z = \sum_i x_i w_i + b$) e aplica uma não linearidade (ReLU, $\max(0,z)$). Empilhar camadas permite representar
relações não lineares e interações entre variáveis sem que o analista as especifique. A camada de saída é **linear** (regressão);
aqui ela tem **H neurônios**, um por horizonte. Os pesos são ajustados por *backpropagation* + Adam minimizando o **MAE** (`L1Loss`).

**Hipóteses e requisitos de dados.** (i) Entradas numéricas e **padronizadas** (redes são sensíveis à escala — por isso o
`StandardScaler` é ajustado só no treino de cada origem); (ii) a relação features→alvo é razoavelmente estável no tempo;
(iii) as linhas são tratadas como amostras independentes, então **o tempo precisa estar codificado nas features** (lags, janelas,
calendário) — o MLP não enxerga ordem temporal sozinho; (iv) dados suficientes para o número de pesos.

**Vantagens.** Captura não linearidades e interações; aceita muitas exógenas; saída multi-horizonte em um único modelo.
**Limitações.** Sensível a escala e inicialização (resultados variam entre sementes); tende a overfitting em séries curtas/ruidosas;
não extrapola de forma confiável fora da faixa vista no treino (por isso o alvo é log-retorno, não o preço); é caixa-preta;
custo de re-treino a cada origem no walk-forward.

**Hiperparâmetros (papel e efeito).**

| Hiperparâmetro | Efeito | Faixa testada |
|---|---|---|
| camadas / neurônios | capacidade do modelo (mais = mais flexível, mais overfitting) | (32), (64), (64,64), (128,64), (128,64,32) |
| dropout | regulariza desligando neurônios aleatórios no treino | 0 – 0,3 |
| learning rate | tamanho do passo do Adam; alto = instável, baixo = lento | 3e-4 – 3e-3 |
| weight decay | penalização L2 dos pesos; reduz overfitting | 0 – 1e-3 |
| batch size | ruído do gradiente e velocidade | 32 – 128 |
| peso por horizonte | `True`: loss = L1 na escala original (alinhada ao MAE em USD); `False`: L1 padronizado uniforme | {True, False} |
| épocas / paciência | quando parar; o *early stopping* evita treinar demais | máx. 150–300, paciência 20 |

**Otimização.** Busca aleatória com **triagem em 2 etapas** (todas as configs em origens esparsas → finalistas na validação completa, com média de `N_SEEDS_TUNE` redes e erro-padrão do MAE); o teste não é tocado.
**Estabilidade.** O MLP final é a **média de `N_SEEDS_FINAL` redes** com sementes diferentes, o que reduz a variância de inicialização e de early stopping.
**Perda.** L1 ponderado por horizonte (`wₖ ∝ σₖ`) = L1 na escala original, coerente com a métrica do trabalho.
**Importância.** Permutation Importance no teste (embaralha cada feature e mede a piora do MAE).

**Desempenho nas 5 bases e explicações.** *(preencher após rodar todas as bases: em quais bases o MLP venceu/perdeu e por quê —
sazonalidade forte favorece SARIMAX/Holt-Winters; muitas exógenas informativas e relações não lineares favorecem RF/MLP;
séries muito ruidosas aproximam todos do Naive.)*

## 13. Persistência dos resultados e consolidação entre bases (item 6)

Cada execução grava `resultados/mae_<BASE>.csv`, `resultados/previsoes_<BASE>.csv` e `resultados/meta_<BASE>.json`
(hiperparâmetros, tempos, features). Depois de rodar as 5 bases (trocando a configuração da seção 0), a função abaixo
gera a tabela de MAE por base, as **vitórias** e a **posição média** de cada modelo.

In [ ]:
import glob
tab_out = tab.reset_index(); tab_out.insert(0, "base", BASE_NAME)
tab_out["referencia"] = tab_out["modelo"].str.contains("Naive")
tab_out.to_csv(f"{RESULTS_DIR}/mae_{BASE_NAME}.csv", index=False)
pd.concat([d.assign(modelo=m, base=BASE_NAME) for m, d in PRED.items()]).to_csv(f"{RESULTS_DIR}/previsoes_{BASE_NAME}.csv", index=False)
tab_lb.to_csv(f"{RESULTS_DIR}/ljungbox_{BASE_NAME}.csv")
tab_dm.to_csv(f"{RESULTS_DIR}/significancia_{BASE_NAME}.csv", index=False)
tab_abl.to_csv(f"{RESULTS_DIR}/ablacao_{BASE_NAME}.csv", index=False)
for nome_t, t_ in (("ano", tab_ano), ("volatilidade", tab_vol), ("tendencia", tab_tend)): t_.to_csv(f"{RESULTS_DIR}/regime_{nome_t}_{BASE_NAME}.csv")
tab_dir.to_csv(f"{RESULTS_DIR}/direcao_{BASE_NAME}.csv")
json.dump({"base": BASE_NAME, "H": H, "STEP": STEP, "N_TEST": N_TEST, "N_VAL": N_VAL, "origens_teste": len(ORIGINS_TEST),
           "teste": [str(dates[TEST_START].date()), str(dates[-1].date())], "hiperparametros": BEST,
           "tempo_s": TEMPO, "tempo_ablacao_s": TEMPO_ABL, "features": FEATURES, "modelo_em_log": MODEL_LOG, "sarimax_exog": SX_EXOG_MODE,
           "n_seeds_final": N_SEEDS_FINAL, "n_seeds_tuning": N_SEEDS_TUNE, "N_VAL": N_VAL, "STEP_VAL": STEP_VAL, "linhas_removidas_sem_pregao": int(stale.sum())},
          open(f"{RESULTS_DIR}/meta_{BASE_NAME}.json", "w"), indent=2, default=str)

def consolidar_resultados(pasta=RESULTS_DIR):
    arqs = sorted(glob.glob(f"{pasta}/mae_*.csv"))
    todos = pd.concat([pd.read_csv(a) for a in arqs]); todos = todos[~todos["referencia"]]
    todos["posicao"] = todos.groupby("base")["MAE"].rank(method="min")
    por_base = todos.pivot(index="base", columns="modelo", values="MAE")
    resumo = todos.groupby("modelo").agg(vitorias=("posicao", lambda s: int((s == 1).sum())),
                                         posicao_media=("posicao", "mean"), bases=("base", "nunique")).sort_values("posicao_media")
    return por_base, resumo

por_base, resumo = consolidar_resultados()
print("MAE por base (NÃO somar/médiar entre bases — escalas diferentes):"); print(por_base.round(4).to_string())
print("\nVitórias e posição média:"); print(resumo.round(3).to_string())

## 14. O que ainda depende do grupo (checklist de entrega — item 13)

- [ ] Rodar as **5 bases** (trocar só a configuração da seção 0) e reexecutar a seção 13 para as vitórias/posição média.
- [ ] Escrever as interpretações (STL/tendência, resíduos, importância, MLP nas 5 bases) — o notebook gera as tabelas e gráficos, **a narrativa é do grupo**.
- [ ] Citar fontes das bases, bibliotecas (incl. **pmdarima**, Smith et al.; método stepwise de Hyndman & Khandakar, 2008) e referências (item 15) e registrar o uso de IA com revisão crítica.
- [ ] Exportar o relatório: `jupyter nbconvert --to html <notebook>.ipynb` (HTML) e imprimir/converter para PDF.
- [ ] Preencher o **registro diário de demandas** (modelo em `registro_demandas_template.csv`).
- [ ] Compactar tudo em `Grupo_XX_Trabalho_Series_Temporais.zip` (PDF, HTML, `.ipynb`, códigos, bases, `resultados/`, registro de demandas).